In [2]:
import pandas as pd
import sys
sys.path.insert(0, "..")        # Search for utils.py module on project's root
from utils import fill_empty_rows, import_datasets_to_db, create_file_path

In [3]:
# Read the .csv file
df_isuzu = pd.read_csv(create_file_path("isuzuR.csv"))

In [4]:
# Check the cols available
df_isuzu.columns

Index(['ModType', 'Hardware', 'Isuzu PN', 'GM PN', 'BCC', 'GM Serv №',
       'Sticker Description', 'VIN Example', 'Year', 'Make & Mdl', 'Engine',
       'Trans.', 'EPC Part Number', 'Hardware ID'],
      dtype='object')

In [5]:
# Print the head
df_isuzu.head(2)

,ModType,Hardware,Isuzu PN,GM PN,BCC,GM Serv №,Sticker Description,VIN Example,Year,Make & Mdl,Engine,Trans.,EPC Part Number,Hardware ID
0,IS-U38B,U38B-63A,8162152393,16215239,BUWP,16205463,3.2S AUT PCM,NaN,1996,Isuzu Rodeo,3.2L,NaN,NaN,892809268
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1996,Honda Passport,3.2L,NaN,NaN,892809268


In [6]:
# Get the index with empty rows
df_isuzu_empty_rows_index = df_isuzu[df_isuzu['ModType'].isna() == True].index

In [7]:
# Drop the empty rows
df_isuzu_drop_empty = df_isuzu.drop(df_isuzu_empty_rows_index)

In [8]:
# Drop the repeated columns
df_isuzu_repeated_cols_index = df_isuzu_drop_empty[df_isuzu_drop_empty['ModType'] == 'ModType'].index

In [9]:
# Drop the repeated cols
df_isuzu_repeated_cols_dropped = df_isuzu_drop_empty.drop(df_isuzu_repeated_cols_index)

In [10]:
# Remove rows where all cols have null (NaN) values,
# keeping only rows with at least one valid (non-null) entry.
df_isuzu_rm_null_rows = df_isuzu_repeated_cols_dropped[~df_isuzu_repeated_cols_dropped.isnull().all(axis=1)]

In [11]:
# Keep only rows where more than one col has a non-null value.
# This removes rows with only a single non-null entry, ensuring each row has multiple valid data points.
df_isuzu_rm_one_null_row = df_isuzu_rm_null_rows[df_isuzu_rm_null_rows.notnull().sum(axis=1) != 1]

In [12]:
# Show cols
df_isuzu_rm_one_null_row.columns

Index(['ModType', 'Hardware', 'Isuzu PN', 'GM PN', 'BCC', 'GM Serv №',
       'Sticker Description', 'VIN Example', 'Year', 'Make & Mdl', 'Engine',
       'Trans.', 'EPC Part Number', 'Hardware ID'],
      dtype='object')

# Cols to keep:
- Hardware
- ModType
- Isuzu PN
- GM PN
- GM Serv №

In [13]:
# List with cols to keep
cols_to_keep = [
    "Hardware", 
    "ModType",
    "Isuzu PN",
    "GM PN",
    "GM Serv №"
]

In [14]:
# Create a df with needed cols
df_isuzu_needed_cols = df_isuzu_rm_one_null_row[cols_to_keep]

In [15]:
# Call the function to fill out null values
df_isuzu_fill_nulls = fill_empty_rows(df_isuzu_needed_cols)

e:\Language_Projects\Language_Projects\Python\Flagship_1\part-number-datasets-cleaning\notebooks\..\utils.py:19: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[col] = df[col].fillna("Not Available")


In [16]:
# Show the available cols
df_isuzu_fill_nulls.columns

Index(['Hardware', 'ModType', 'Isuzu PN', 'GM PN', 'GM Serv №'], dtype='object')

In [18]:
# Map to rename 'Hardware' col
rn_cols_map = {
    "Hardware": "hardware",
    "ModType":  "pn1",
    "Isuzu PN": "pn2",
    "GM PN": "pn3",
    "GM Serv №": "pn4"
}

In [20]:
# Rename 'Hardware' col
df_isuzu_rn_cols = df_isuzu_fill_nulls.rename(columns=rn_cols_map)

In [21]:
# Check df info
df_isuzu_rn_cols.info()

<class 'pandas.core.frame.DataFrame'>
Index: 101 entries, 0 to 148
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   hardware  101 non-null    object
 1   pn1       101 non-null    object
 2   pn2       101 non-null    object
 3   pn3       101 non-null    object
 4   pn4       101 non-null    object
dtypes: object(5)
memory usage: 4.7+ KB


In [22]:
# Call the function to import the df cleaned to the postgreSQLdb
import_datasets_to_db(df_isuzu_rn_cols, 'part_numbers_isuzu')

101